# EfficientNetB4 + Frequency-Spatial Dual Attention (FSDA) — Garlic Disease Classification

**Novel Architecture Contribution:** Frequency-Spatial Dual Attention (FSDA) block inserted between EfficientNetB4 backbone and classification head.

| Item | Value |
|---|---|
| **Architecture** | EfficientNetB4 + FSDA Block (novel) |
| **Input size** | 380 × 380 × 3 |
| **FSDA — Freq Branch** | FFT2D → log-magnitude → channel attention (WHAT frequency patterns) |
| **FSDA — Spatial Branch** | Avg+Max pool → Conv2D(7×7) → spatial attention (WHERE to look) |
| **Fusion** | Element-wise addition of two attended feature maps |
| **Head** | GAP → BN → Dense(256,relu) → Dropout → softmax |
| **Optimizer** | Adam + ExponentialDecay (lr=1e-4) |
| **Loss** | CategoricalCrossentropy (label_smoothing=0.15) |
| **Regularisation** | Dropout 0.5 + L2 1e-5 + Class-weight balancing |
| **Strategy** | 3-seed independent runs for statistical robustness |

---

## Why FSDA?

Garlic diseases manifest as **texture anomalies** (spots, lesions, discolouration) that correspond to **specific spatial frequency patterns**:
- **High-frequency components** → sharp disease boundaries, spot edges
- **Mid-frequency components** → lesion texture, surface roughness  
- **Low-frequency components** → overall colour/illumination changes

Standard CNN+GAP treats all spatial positions and all channels equally. FSDA provides two complementary re-weightings:
1. **Frequency Channel Attention** — *which channels carry disease-relevant frequency information?*
2. **Spatial Attention** — *where in the image are the disease lesions located?*

In [ ]:
# ========== 1. IMPORTS ========== #
import os
import csv
import time
import random
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm_lib
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Activation, Input,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    top_k_accuracy_score, roc_curve, auc,
    cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# ========== 2. GPU CONFIG ========== #
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", len(gpus))
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as e:
        print(e)

# ========== 3. MIXED PRECISION ========== #
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype :", tf.keras.mixed_precision.global_policy().compute_dtype)
print("Variable dtype:", tf.keras.mixed_precision.global_policy().variable_dtype)

In [ ]:
# ========== CUSTOM LOSS: LDAM LOSS ========== #
# Cao et al., 2019. Label-Distribution-Aware Margin Loss.
# Assigns larger decision margins to minority classes: Δ_j = C / n_j^{1/4}
# Applied via logit adjustment: logit_j -= Δ_j (true class only).
# Uses log-trick to recover approximate logits from softmax probabilities.
# Requires samples_per_class (computed from training split before model build).

class LDAMLoss(tf.keras.losses.Loss):
    def __init__(self, samples_per_class, max_margin=0.5, s=30.0, **kwargs):
        super().__init__(**kwargs)
        self.max_margin         = max_margin
        self.s                  = s
        self._samples_per_class = list(samples_per_class)
        n       = np.array(samples_per_class, dtype=np.float32)
        margins = max_margin / np.power(n, 0.25)
        self.margins = tf.constant(margins, dtype=tf.float32)

    def call(self, y_true, y_pred):
        y_true  = tf.cast(y_true, tf.float32)
        y_pred  = tf.cast(y_pred, tf.float32)
        logits  = tf.math.log(tf.clip_by_value(y_pred, 1e-7, 1.0))
        margins = tf.reduce_sum(y_true * self.margins, axis=-1, keepdims=True)
        loss    = tf.keras.losses.categorical_crossentropy(
            y_true, (logits - margins) * self.s, from_logits=True)
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'samples_per_class': self._samples_per_class,
            'max_margin': self.max_margin, 's': self.s,
        })
        return cfg

print("LDAMLoss defined  (max_margin=0.5, s=30.0)")


In [ ]:
# ========== 4. EXPERIMENT CONFIGURATION ========== #

# ── STRATEGY ──────────────────────────────────────────────────────────────
STRATEGY_KEY    = "finetune_top5_fsda_ldam"
STRATEGY_LABEL  = "EfficientNetB4 + FSDA (LDAM Loss)"
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
USE_AUG         = True
LR              = 1e-4
# ──────────────────────────────────────────────────────────────────────────

# ── FSDA HYPERPARAMETERS ──────────────────────────────────────────────────
# reduction: bottleneck ratio for frequency channel attention MLP
# Smaller = more capacity; larger = more regularisation.
FSDA_REDUCTION    = 16   # C // 16 hidden units in freq attention MLP
FSDA_SPATIAL_KS   = 7    # kernel size for spatial attention conv (3, 5, or 7)
# ──────────────────────────────────────────────────────────────────────────

# --- Paths ---
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# --- Model ---
INPUT_SHAPE = (380, 380, 3)
BATCH_SIZE  = 32    # smaller batch: FSDA adds memory overhead vs plain GAP
EPOCHS      = 30

# --- Shared hyperparameters ---
LABEL_SMOOTHING = 0.15  # kept for reference; not used by LDAM Loss
DROPOUT_RATE    = 0.5
PATIENCE        = 12

# --- Multi-run for statistical robustness ---
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]

# --- Performance knobs ---
AUTOTUNE = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

# --- Result storage ---
all_runs_results = []

print(f"Strategy   : {STRATEGY_LABEL}  (key={STRATEGY_KEY})")
print(f"Loss       : LDAMLoss  (max_margin=0.5, s=30.0)")
print(f"Unfreeze   : {UNFREEZE_BLOCKS}  |  Augmentation: {USE_AUG}  |  LR: {LR}")
print(f"Dataset    : {DATA_DIR}")
print(f"Output dir : {BASE_RESULT_DIR}")
print(f"Input shape: {INPUT_SHAPE}")
print(f"Batch size : {BATCH_SIZE}")
print(f"N_RUNS     : {N_RUNS}  |  Seeds: {RANDOM_SEEDS[:N_RUNS]}  (3-seed avg for statistical robustness)")
print(f"FSDA       : reduction={FSDA_REDUCTION}  spatial_ks={FSDA_SPATIAL_KS}")
print(f"XLA JIT    : ON")


In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    """Selectively unfreeze EfficientNetB4 blocks (BN always frozen)."""
    base.trainable = False
    if unfreeze_blocks == "all":
        for layer in base.layers:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True
    elif isinstance(unfreeze_blocks, list) and len(unfreeze_blocks) > 0:
        for layer in base.layers:
            for block_num in unfreeze_blocks:
                if layer.name.startswith(f"block{block_num}"):
                    if not isinstance(layer, tf.keras.layers.BatchNormalization):
                        layer.trainable = True
                    break
    trainable = sum(1 for l in base.layers if l.trainable)
    total     = len(base.layers)
    print(f"  [{STRATEGY_LABEL}] {trainable}/{total} base layers trainable (BN always frozen)")


def _collect_samples(split_dir, class_to_idx):
    """Walk split_dir -> (abs_paths, int_labels, rel_filenames), sorted."""
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None, use_aug=True):
    """Build GPU-optimised tf.data pipelines (one-hot labels for training)."""
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes  = len(class_names)
    h, w         = input_shape[:2]

    def load_and_preprocess(path, label):
        raw   = tf.io.read_file(path)
        img   = tf.image.decode_jpeg(raw, channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = tf.cast(img, tf.float32)
        img   = efficientnet_preprocess(img)
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False, apply_augmentation=False):
        sdir               = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if apply_augmentation:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training)
        ds = ds.prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl  = _make_split('train', training=True,  apply_augmentation=use_aug)
    val_ds,   n_val,   _,           _          = _make_split('val',   training=False, apply_augmentation=False)
    test_ds,  n_test,  test_fnames, test_lbl   = _make_split('test',  training=False, apply_augmentation=False)

    cw      = class_weight.compute_class_weight(
        'balanced', classes=np.unique(train_lbl), y=train_lbl)
    cw_dict = dict(enumerate(cw))

    meta = SimpleNamespace(
        class_names       = class_names,
        class_to_idx      = class_to_idx,
        num_classes       = num_classes,
        test_filenames    = test_fnames,
        test_classes      = np.array(test_lbl),
        n_train           = n_train,
        n_val             = n_val,
        n_test            = n_test,
        class_weight_dict = cw_dict,
    )
    aug_info = "ON" if use_aug else "OFF"
    print(f"  Augmentation: {aug_info}  |  train={n_train}  val={n_val}  test={n_test}")
    return train_ds, val_ds, test_ds, meta


print("Helper functions defined.")

In [ ]:

# ========== 6. FSDA ARCHITECTURE ========== #
#
#  EfficientNetB4 backbone (fine-tuned blocks 3-7)
#       |  feature_map  (B, 12, 12, 1792)
#       |
#  +----+----+
#  |         |
#  v         v
#  FrequencyChannelAttention   SpatialAttention
#  (FFT -> which channels)     (where to look)
#  |         |
#  +----+----+
#       | element-wise add (float32) + BN -> cast back to input dtype
#       v
#  GlobalAveragePooling2D
#       v
#  BN -> Dense(256,relu) -> Dropout -> softmax


class FrequencyChannelAttention(tf.keras.layers.Layer):
    """
    Frequency-domain Channel Attention.
    All internal computation is done in float32 to avoid mixed-precision
    dtype conflicts. The output is cast back to the caller's input dtype.
    """
    def __init__(self, reduction=16, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = input_shape[-1]
        r = max(C // self.reduction, 8)
        self.fc1 = Dense(r, use_bias=False, dtype='float32',
                         name=f'{self.name}_fc1')
        self.fc2 = Dense(C, use_bias=False, dtype='float32',
                         name=f'{self.name}_fc2')
        self.fc1.build((None, C))
        self.fc2.build((None, r))
        super().build(input_shape)

    def call(self, x, training=False):
        x_f32 = tf.cast(x, tf.float32)
        x_t       = tf.transpose(x_f32, [0, 3, 1, 2])
        x_complex = tf.complex(x_t, tf.zeros_like(x_t))
        x_fft     = tf.signal.fft2d(x_complex)
        mag       = tf.math.log1p(tf.abs(x_fft))
        freq_desc = tf.reduce_mean(mag, axis=[2, 3])
        attn = tf.nn.relu(self.fc1(freq_desc))
        attn = tf.nn.sigmoid(self.fc2(attn))
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        out  = x_f32 * attn
        return tf.cast(out, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg['reduction'] = self.reduction
        return cfg


class FSDABlock(tf.keras.layers.Layer):
    """
    Frequency-Spatial Dual Attention (FSDA) Block.
    Returns: (fused_features, spatial_attn_map)
    """
    def __init__(self, reduction=16, spatial_kernel=7, **kwargs):
        super().__init__(**kwargs)
        self.reduction      = reduction
        self.spatial_kernel = spatial_kernel

    def build(self, input_shape):
        self.freq_attn = FrequencyChannelAttention(
            self.reduction, name=f'{self.name}_freq_attn')
        self.sp_conv = Conv2D(
            1, self.spatial_kernel, padding='same', use_bias=False,
            kernel_initializer='glorot_uniform', dtype='float32',
            name=f'{self.name}_sp_conv')
        self.bn = BatchNormalization(dtype='float32',
                                     name=f'{self.name}_bn')
        self.freq_attn.build(input_shape)
        sp_input_shape = tuple(input_shape[:-1]) + (2,)
        self.sp_conv.build(sp_input_shape)
        self.bn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        freq_out = tf.cast(self.freq_attn(x, training=training), tf.float32)
        x_f32    = tf.cast(x, tf.float32)
        avg_pool = tf.reduce_mean(x_f32, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x_f32,  axis=-1, keepdims=True)
        sp_attn  = tf.nn.sigmoid(
            self.sp_conv(tf.concat([avg_pool, max_pool], axis=-1))
        )
        spatial_out = x_f32 * sp_attn
        fused = freq_out + spatial_out
        fused = self.bn(fused, training=training)
        fused = tf.cast(fused, input_dtype)
        return fused, sp_attn

    def compute_output_spec(self, x, training=False):
        """Bypass tf.signal.fft2d symbolic tracing for Keras 3 model loading."""
        import keras
        sp_shape = tuple(x.shape[:-1]) + (1,)
        return (
            keras.KerasTensor(x.shape,   dtype=x.dtype),
            keras.KerasTensor(sp_shape,  dtype='float32'),
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction, 'spatial_kernel': self.spatial_kernel})
        return cfg


# Custom objects dict for load_model calls
CUSTOM_OBJECTS = {
    'FrequencyChannelAttention': FrequencyChannelAttention,
    'FSDABlock':                 FSDABlock,
    'LDAMLoss':                  LDAMLoss,
}


def build_fsda_model(input_shape, num_classes, steps_per_epoch, samples_per_class):
    base = EfficientNetB4(weights='imagenet', include_top=False,
                          input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    feat_map = base.output

    attended, sp_attn_map = FSDABlock(
        reduction=FSDA_REDUCTION,
        spatial_kernel=FSDA_SPATIAL_KS,
        name='fsda',
    )(feat_map)

    x   = GlobalAveragePooling2D(name='gap')(attended)
    x   = BatchNormalization(name='head_bn')(x)
    x   = Dense(256, activation='relu',
                kernel_regularizer=l2(1e-5), name='head_dense')(x)
    x   = Dropout(DROPOUT_RATE, name='head_dropout')(x)
    out = Dense(num_classes, activation='softmax',
                dtype='float32', name='predictions')(x)

    model = Model(inputs=base.input, outputs=out,
                  name='EfficientNetB4_FSDA')

    vis_model = Model(inputs=base.input, outputs=[out, sp_attn_map],
                      name='EfficientNetB4_FSDA_vis')

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR,
        decay_steps=steps_per_epoch * 5,
        decay_rate=0.9,
        staircase=True,
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=LDAMLoss(samples_per_class, max_margin=0.5, s=30.0),
        metrics=['accuracy'],
    )
    return model, vis_model


print("FSDA architecture defined.")
print("  - FrequencyChannelAttention: fc1/fc2 explicitly built in build()")
print("  - FSDABlock: sp_conv + bn explicitly built in build() → Keras 3 safe")
print("  - FSDABlock.compute_output_spec: bypasses FFT symbolic tracing")
print("  - build_fsda_model: returns (model, vis_model)")
print("  - Loss: LDAMLoss(max_margin=0.5, s=30.0)")


In [ ]:
# ========== 7. MULTI-RUN TRAINING ========== #

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{N_RUNS}  --  seed={seed}")
    print(f" Strategy : {STRATEGY_LABEL}  |  LR={LR}  |  Aug={USE_AUG}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    # ── Build datasets ─────────────────────────────────────────────────────
    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed, use_aug=USE_AUG)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    # ── Compute samples per class from training split ──────────────────────
    samples_per_class = np.array([
        len([f for f in os.listdir(os.path.join(DATA_DIR, 'train', cn))
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))])
        for cn in meta.class_names
    ])
    print(f"  Samples/class: {dict(zip(meta.class_names, samples_per_class))}")

    # ── Build FSDA model ───────────────────────────────────────────────────
    model, vis_model = build_fsda_model(INPUT_SHAPE, meta.num_classes, steps_per_epoch, samples_per_class)
    print(f"  Trainable params: {model.count_params():,}")

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv'), append=False),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ]

    # NOTE: class_weight intentionally omitted.
    # LDAMLoss assigns larger margins to minority classes (Δ_j ∝ 1/n_j^0.25),
    # already encoding per-class imbalance. Passing class_weight on top would
    # double-count imbalance and causes TypeError with LearningRateSchedule
    # in Keras 3.
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    # ── Learning curves ────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, keys, title in zip(
        axes,
        [('accuracy', 'val_accuracy'), ('loss', 'val_loss')],
        ['Accuracy', 'Loss']
    ):
        ax.plot(history.history[keys[0]], label='Train')
        ax.plot(history.history[keys[1]], label='Validation')
        ax.set_title(f'{title} -- Run {run_idx+1} (seed={seed})')
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'learning_curve.png'), dpi=300)
    plt.show()

    # ── Evaluate on test set ───────────────────────────────────────────────
    best_model = load_model(
        os.path.join(RESULT_DIR, 'best_model.keras'),
        custom_objects=CUSTOM_OBJECTS,
    )
    pred_probs = best_model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes
    class_names = meta.class_names

    report = classification_report(
        y_true_run, y_pred_run,
        target_names=class_names, output_dict=True, digits=4)
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(
            y_true_run, y_pred_run, target_names=class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=class_names, yticklabels=class_names,
                cmap='Blues', ax=ax)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix -- Run {run_idx+1} (seed={seed})')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300)
    plt.close()

    test_acc = np.mean(y_pred_run == y_true_run)

    # ── Store results ──────────────────────────────────────────────────────
    all_runs_results.append({
        'run':               run_idx + 1,
        'seed':              seed,
        'accuracy':          test_acc,
        'precision':         report['weighted avg']['precision'],
        'recall':            report['weighted avg']['recall'],
        'f1_score':          report['weighted avg']['f1-score'],
        'per_class_metrics': {c: {'precision': report[c]['precision'],
                                   'recall':    report[c]['recall'],
                                   'f1-score':  report[c]['f1-score']}
                               for c in class_names},
        'result_dir':        RESULT_DIR,
        'history':           history.history,
        'y_true':            y_true_run,
        'y_pred':            y_pred_run,
        'pred_probs':        pred_probs,
        'class_names':       class_names,
        'test_filenames':    meta.test_filenames,
        'n_train':           meta.n_train,
        'n_val':             meta.n_val,
        'n_test':            meta.n_test,
        'best_model_path':   os.path.join(RESULT_DIR, 'best_model.keras'),
    })

    print(f"\n  Acc={test_acc:.4f}  "
          f"P={report['weighted avg']['precision']:.4f}  "
          f"R={report['weighted avg']['recall']:.4f}  "
          f"F1={report['weighted avg']['f1-score']:.4f}")

    tf.keras.backend.clear_session()

# ── Save per-run summary CSV ───────────────────────────────────────────────
summary_path = os.path.join(BASE_RESULT_DIR, 'strategy_summary.csv')
with open(summary_path, 'w', newline='') as csvf:
    fieldnames = ['strategy_key', 'strategy_label', 'unfreeze_blocks', 'lr',
                  'use_aug', 'fsda_reduction', 'fsda_spatial_ks',
                  'run', 'seed', 'accuracy', 'precision', 'recall', 'f1_score']
    writer = csv.DictWriter(csvf, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_runs_results:
        writer.writerow({
            'strategy_key':    STRATEGY_KEY,
            'strategy_label':  STRATEGY_LABEL,
            'unfreeze_blocks': str(UNFREEZE_BLOCKS),
            'lr':              LR,
            'use_aug':         USE_AUG,
            'fsda_reduction':  FSDA_REDUCTION,
            'fsda_spatial_ks': FSDA_SPATIAL_KS,
            'run':             r['run'],
            'seed':            r['seed'],
            'accuracy':        r['accuracy'],
            'precision':       r['precision'],
            'recall':          r['recall'],
            'f1_score':        r['f1_score'],
        })

print("\n" + "="*70)

print(f" ALL {N_RUNS} RUN(S) COMPLETED -- {STRATEGY_LABEL}")

print(f" Saved summary -> {summary_path}")print("="*70)

---
## Section 2 — Results Aggregation & Scientific Reports

In [ ]:
# ========== 8. AGGREGATE RESULTS ========== #

accuracies = [r['accuracy']  for r in all_runs_results]
precisions = [r['precision'] for r in all_runs_results]
recalls    = [r['recall']    for r in all_runs_results]
f1_scores  = [r['f1_score']  for r in all_runs_results]

overall_stats = {
    'Accuracy':  {'mean': np.mean(accuracies), 'std': np.std(accuracies), 'values': accuracies},
    'Precision': {'mean': np.mean(precisions), 'std': np.std(precisions), 'values': precisions},
    'Recall':    {'mean': np.mean(recalls),    'std': np.std(recalls),    'values': recalls},
    'F1-Score':  {'mean': np.mean(f1_scores),  'std': np.std(f1_scores),  'values': f1_scores},
}

print("OVERALL METRICS ACROSS ALL RUNS:")
print("-" * 60)
for metric_name, stats in overall_stats.items():
    print(f"{metric_name:12s}: {stats['mean']:.4f} +/- {stats['std']:.4f}")
    print(f"              Individual: {[f'{v:.4f}' for v in stats['values']]}")
print("-" * 60)

class_names = list(all_runs_results[0]['per_class_metrics'].keys())
per_class_stats = {}
for cn in class_names:
    per_class_stats[cn] = {}
    for metric in ['precision', 'recall', 'f1-score']:
        values = [r['per_class_metrics'][cn][metric] for r in all_runs_results]
        per_class_stats[cn][metric] = {
            'mean': np.mean(values), 'std': np.std(values), 'values': values
        }

# ── Summary table ──────────────────────────────────────────────────────────
overall_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Mean':   [overall_stats[m]['mean'] for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']],
    'Std':    [overall_stats[m]['std']  for m in ['Accuracy', 'Precision', 'Recall', 'F1-Score']],
})
overall_df['Mean +/- Std'] = overall_df.apply(
    lambda row: f"{row['Mean']:.4f} +/- {row['Std']:.4f}", axis=1)
for i in range(len(all_runs_results)):
    overall_df[f'Run {i+1}'] = [
        accuracies[i], precisions[i], recalls[i], f1_scores[i]]

print("\nSUMMARY TABLE")
print("=" * 70)
run_cols = [f'Run {i+1}' for i in range(len(all_runs_results))]
print(overall_df[['Metric', 'Mean +/- Std'] + run_cols].to_string(index=False))
print("=" * 70)
overall_df.to_csv(os.path.join(BASE_RESULT_DIR, 'overall_metrics_summary.csv'), index=False)
print(f"\nSaved -> {BASE_RESULT_DIR}/overall_metrics_summary.csv")

In [ ]:
# ========== 9. PER-CLASS METRICS BAR CHART ========== #

metrics_to_plot = ['precision', 'recall', 'f1-score']
x = np.arange(len(class_names))
bar_w = 0.25
colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(max(10, len(class_names) * 1.5), 5))
for i, metric in enumerate(metrics_to_plot):
    means = [per_class_stats[cn][metric]['mean'] for cn in class_names]
    stds  = [per_class_stats[cn][metric]['std']  for cn in class_names]
    ax.bar(x + i * bar_w, means, bar_w, yerr=stds, label=metric.capitalize(),
           color=colors[i], alpha=0.85, capsize=4, edgecolor='white')

ax.set_xlabel('Class', fontweight='bold')
ax.set_ylabel('Score', fontweight='bold')
ax.set_title(f'Per-Class Metrics -- {STRATEGY_LABEL}\n(Mean +/- Std over {N_RUNS} runs)',
             fontweight='bold')
ax.set_xticks(x + bar_w)
ax.set_xticklabels(class_names, rotation=30, ha='right')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'per_class_metrics.png'), dpi=300)
plt.show()

In [ ]:
# ========== 10. AGGREGATE CONFUSION MATRIX ========== #

agg_cm = np.zeros((len(class_names), len(class_names)), dtype=int)
for r in all_runs_results:
    agg_cm += confusion_matrix(r['y_true'], r['y_pred'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(agg_cm, annot=True, fmt='d',
            xticklabels=class_names, yticklabels=class_names,
            cmap='Blues', ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title(f'Aggregate Confusion Matrix (raw counts, {N_RUNS} runs)')

# Normalised
cm_norm = agg_cm.astype(float) / agg_cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f',
            xticklabels=class_names, yticklabels=class_names,
            cmap='Blues', ax=axes[1], vmin=0, vmax=1)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].set_title('Aggregate Confusion Matrix (normalised)')

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'agg_confusion_matrix.png'), dpi=300)
plt.show()

In [ ]:
# ========== 11. ROC CURVES (best run) ========== #

best_run = max(all_runs_results, key=lambda r: r['accuracy'])
y_true_b = best_run['y_true']
y_prob_b = best_run['pred_probs']
nc       = len(class_names)

y_bin = label_binarize(y_true_b, classes=list(range(nc)))
fpr_m, tpr_m, _ = roc_curve(y_bin.ravel(), y_prob_b.ravel())
auc_m = auc(fpr_m, tpr_m)

fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.cm.get_cmap('tab10', nc)
for i, cn in enumerate(class_names):
    fpr_i, tpr_i, _ = roc_curve(y_bin[:, i], y_prob_b[:, i])
    ax.plot(fpr_i, tpr_i, color=cmap(i),
            label=f'{cn} (AUC={auc(fpr_i, tpr_i):.3f})', linewidth=1.5)
ax.plot(fpr_m, tpr_m, 'k--', label=f'Macro avg (AUC={auc_m:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'gray', linestyle=':', linewidth=1)
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title(f'ROC Curves -- {STRATEGY_LABEL}\n(Best run: seed={best_run["seed"]})',
             fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'roc_curves.png'), dpi=300)
plt.show()
print(f"Macro-average AUC: {auc_m:.4f}")

---
## Section 3 — FSDA Attention Map Visualization

**This section is unique to the FSDA architecture.**  
The spatial attention map from `FSDABlock` shows WHERE the model focuses within each image — directly interpretable as "disease localization" without needing Grad-CAM.

In [ ]:
# ========== 12. FSDA SPATIAL ATTENTION MAP VISUALIZATION ========== #
# Shows the 12x12 spatial attention map overlaid on the original image.
# Bright regions = where FSDA told the model to focus.
# NOTE: No retraining needed — vis_model is rebuilt from saved weights.

best_run   = max(all_runs_results, key=lambda r: r['accuracy'])
best_model = load_model(
    best_run['best_model_path'], custom_objects=CUSTOM_OBJECTS)

# ── Rebuild vis_model that also outputs the spatial attention map ──────────
# best_model only outputs 'predictions'. To get sp_attn we re-call the FSDA
# layer on its own input tensor (already in the graph), which shares weights.
fsda_layer = best_model.get_layer('fsda')
fsda_in    = fsda_layer.input              # KerasTensor fed into FSDABlock
_, sp_attn_map = fsda_layer(fsda_in)       # re-call → (fused, sp_attn) float32

vis_model = Model(
    inputs  = best_model.input,
    outputs = [best_model.output, sp_attn_map],
    name    = 'vis_model',
)

h, w = INPUT_SHAPE[:2]

# Sample 2 images per class from the test set
test_dir           = os.path.join(DATA_DIR, 'test')
class_names_sorted = sorted(os.listdir(test_dir))
samples_per_class  = 2

sample_imgs, sample_paths, sample_labels = [], [], []
for cn in class_names_sorted:
    cls_dir = os.path.join(test_dir, cn)
    imgs = sorted([
        f for f in os.listdir(cls_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])[:samples_per_class]
    for fname in imgs:
        fpath = os.path.join(cls_dir, fname)
        orig  = tf.image.decode_jpeg(tf.io.read_file(fpath), channels=3)
        orig  = tf.image.resize(orig, [h, w]).numpy().astype(np.uint8)
        sample_imgs.append(orig)
        sample_paths.append(fpath)
        sample_labels.append(cn)

# Batch inference
preprocessed = np.stack([
    efficientnet_preprocess(
        tf.cast(tf.image.resize(
            tf.image.decode_jpeg(tf.io.read_file(p), channels=3),
            [h, w]), tf.float32)).numpy()
    for p in sample_paths
])
preds_vis, attn_maps = vis_model(preprocessed, training=False)
pred_classes = [class_names_sorted[i] for i in np.argmax(preds_vis.numpy(), axis=1)]
attn_maps    = attn_maps.numpy()   # (N, 12, 12, 1) float32

# Plot: original | attention overlay
n_samples = len(sample_imgs)
fig, axes = plt.subplots(n_samples, 2, figsize=(6, n_samples * 2.8))
fig.suptitle(f'FSDA Spatial Attention Maps -- {STRATEGY_LABEL}\n'
             f'(best run, seed={best_run["seed"]})',
             fontsize=11, fontweight='bold')

for idx in range(n_samples):
    orig    = sample_imgs[idx]
    attn    = attn_maps[idx, :, :, 0]                    # (12, 12)
    attn_up = tf.image.resize(
        attn[..., np.newaxis], [h, w]).numpy()[:, :, 0]  # upsample to H x W

    attn_n  = (attn_up - attn_up.min()) / (attn_up.max() - attn_up.min() + 1e-8)
    heatmap = (cm_lib.jet(attn_n)[:, :, :3] * 255).astype(np.uint8)
    overlay = (0.55 * orig + 0.45 * heatmap).astype(np.uint8)

    correct = sample_labels[idx] == pred_classes[idx]
    color   = 'green' if correct else 'red'
    conf    = float(np.max(preds_vis.numpy()[idx]))

    axes[idx, 0].imshow(orig)
    axes[idx, 0].set_title(f'True: {sample_labels[idx]}', fontsize=8)
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(overlay)
    axes[idx, 1].set_title(
        f'Pred: {pred_classes[idx]} ({conf:.2f})', fontsize=8, color=color)
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'fsda_attention_maps.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved FSDA attention maps -> {BASE_RESULT_DIR}/fsda_attention_maps.png")

In [ ]:
# ========== 13. FREQUENCY SPECTRUM VISUALIZATION ========== #
# Shows the FFT magnitude spectrum for sample images per class.
# Motivation: confirms that disease classes have distinct frequency signatures.

fig, axes = plt.subplots(
    len(class_names_sorted), 3,
    figsize=(10, len(class_names_sorted) * 2.5))
fig.suptitle('Frequency Spectrum per Class\n'
             '(FSDA motivation: disease patterns create distinct frequency signatures)',
             fontsize=10, fontweight='bold')

for ci, cn in enumerate(class_names_sorted):
    cls_dir  = os.path.join(test_dir, cn)
    fpath    = os.path.join(cls_dir, sorted(os.listdir(cls_dir))[0])
    orig_rgb = tf.image.resize(
        tf.image.decode_jpeg(tf.io.read_file(fpath), channels=3),
        [h, w]).numpy().astype(np.uint8)

    # Convert to grayscale for FFT display
    gray   = np.mean(orig_rgb, axis=-1)          # (H, W)
    f      = np.fft.fft2(gray)
    fshift = np.fft.fftshift(f)
    mag    = np.log1p(np.abs(fshift))             # log-magnitude

    # Radial average (1D frequency profile)
    cy, cx = np.array(mag.shape) // 2
    Y, X   = np.ogrid[:mag.shape[0], :mag.shape[1]]
    R      = np.sqrt((X - cx)**2 + (Y - cy)**2).astype(int)
    r_max  = min(cy, cx)
    radial = np.array([mag[R == r].mean() if (R == r).any() else 0
                        for r in range(r_max)])

    axes[ci, 0].imshow(orig_rgb)
    axes[ci, 0].set_title(cn, fontsize=8, fontweight='bold')
    axes[ci, 0].axis('off')

    axes[ci, 1].imshow(mag, cmap='inferno')
    axes[ci, 1].set_title('FFT log-magnitude', fontsize=7)
    axes[ci, 1].axis('off')

    axes[ci, 2].plot(radial, linewidth=1.2, color='steelblue')
    axes[ci, 2].set_title('Radial frequency profile', fontsize=7)
    axes[ci, 2].set_xlabel('Frequency (cycles/image)', fontsize=6)
    axes[ci, 2].set_ylabel('Avg magnitude', fontsize=6)
    axes[ci, 2].grid(alpha=0.3)
    axes[ci, 2].tick_params(labelsize=6)

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'frequency_spectra.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved frequency spectra -> {BASE_RESULT_DIR}/frequency_spectra.png")

In [ ]:
# ========== 14. t-SNE FEATURE VISUALIZATION ========== #

import gc

# Clear GPU memory BEFORE loading the model (not after)
gc.collect()
tf.keras.backend.clear_session()

best_run   = max(all_runs_results, key=lambda r: r['accuracy'])
best_model = load_model(
    best_run['best_model_path'], custom_objects=CUSTOM_OBJECTS)

# Feature extractor: output of GAP (after FSDA attention)
gap_model = Model(
    inputs  = best_model.input,
    outputs = best_model.get_layer('gap').output,
)

# Extract test features with smaller batch size to avoid OOM
_, _, test_ds_tsne, meta_tsne = create_tf_datasets(
    DATA_DIR, INPUT_SHAPE, 8, seed=best_run['seed'], use_aug=False)

feats_list, labels_list = [], []
for batch_x, batch_y in test_ds_tsne:
    try:
        feats_batch = gap_model(batch_x, training=False).numpy()
        feats_list.append(feats_batch)
        labels_list.append(np.argmax(batch_y.numpy(), axis=1))
    except tf.errors.ResourceExhaustedError:
        # If still OOM, process in even smaller chunks
        sub_batch_size = 2
        for i in range(0, batch_x.shape[0], sub_batch_size):
            sub_x = batch_x[i:i+sub_batch_size]
            sub_y = batch_y[i:i+sub_batch_size]
            feats_batch = gap_model(sub_x, training=False).numpy()
            feats_list.append(feats_batch)
            labels_list.append(np.argmax(sub_y.numpy(), axis=1))

feats  = np.concatenate(feats_list)
labels = np.concatenate(labels_list)

# Free GPU memory after feature extraction (t-SNE runs on CPU)
del gap_model, best_model
gc.collect()

# t-SNE
print(f"Running t-SNE on {feats.shape[0]} samples x {feats.shape[1]} dims...")
tsne    = TSNE(n_components=2, perplexity=30, random_state=best_run['seed'],
               n_iter=1000, verbose=0)
coords  = tsne.fit_transform(feats)

fig, ax = plt.subplots(figsize=(9, 7))
cmap    = plt.cm.get_cmap('tab10', len(class_names))
for ci, cn in enumerate(class_names):
    mask = labels == ci
    ax.scatter(coords[mask, 0], coords[mask, 1],
               color=cmap(ci), label=cn, alpha=0.7, s=20, edgecolors='none')
ax.set_title(f't-SNE of FSDA-attended GAP features\n'
             f'{STRATEGY_LABEL} (seed={best_run["seed"]})',
             fontweight='bold')
ax.legend(loc='upper right', fontsize=8, markerscale=2)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'tsne.png'), dpi=300)
plt.show()
print(f"Saved t-SNE visualization -> {BASE_RESULT_DIR}/tsne.png")

In [ ]:
# ========== 15. GRAD-CAM++ (on EfficientNetB4 top conv features) ========== #

import gc

def gradcam_pp(model, img_array, layer_name, class_idx=None):
    """
    Grad-CAM++ heatmap. Works with mixed-precision models (float16 layers).

    Key: watch conv_out_orig (float16) explicitly so the tape records
    loss → conv_out_orig gradients. Cast to float32 AFTER gradient computation.
    """
    grad_model = Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(layer_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        tape.watch(img_array)
        conv_out_orig, preds = grad_model(img_array, training=False)
        # Explicitly watch the layer output (may be float16) so the tape
        # can compute d(loss)/d(conv_out_orig) even in mixed-precision.
        tape.watch(conv_out_orig)
        preds_f32 = tf.cast(preds, tf.float32)
        if class_idx is None:
            class_idx = tf.argmax(preds_f32[0])
        loss = preds_f32[:, class_idx]

    # Gradient w.r.t. the original (possibly float16) layer output
    grads    = tape.gradient(loss, conv_out_orig)
    # Cast both to float32 for all subsequent arithmetic
    grads    = tf.cast(grads,         tf.float32)
    conv_out = tf.cast(conv_out_orig, tf.float32)

    grads_sq  = grads ** 2
    grads_cu  = grads ** 3
    alpha_num = grads_sq
    alpha_den = 2 * grads_sq + conv_out * grads_cu
    alpha_den = tf.where(alpha_den == 0, tf.ones_like(alpha_den), alpha_den)
    alpha     = alpha_num / alpha_den
    relu_grad = tf.nn.relu(loss * grads)
    weights   = tf.reduce_mean(alpha * relu_grad, axis=(1, 2))[0]
    cam       = tf.reduce_sum(conv_out[0] * weights, axis=-1).numpy()
    cam       = np.maximum(cam, 0)
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam


# Clear GPU memory before loading model
gc.collect()
tf.keras.backend.clear_session()

best_run   = max(all_runs_results, key=lambda r: r['accuracy'])
best_model = load_model(
    best_run['best_model_path'], custom_objects=CUSTOM_OBJECTS)

# Sub-layers inside FSDABlock are NOT exposed as top-level model layers.
# 'top_activation' is the last EfficientNetB4 feature map before FSDA.
target_layer = 'top_activation'

n_display = min(3, len(sample_imgs))
fig, axes  = plt.subplots(n_display, 3, figsize=(11, n_display * 3.2))
fig.suptitle(f'Grad-CAM++ on EfficientNetB4 top conv features\n{STRATEGY_LABEL}',
             fontsize=10, fontweight='bold')

for idx in range(n_display):
    orig   = sample_imgs[idx]
    p      = sample_paths[idx]
    prep   = efficientnet_preprocess(
        tf.cast(tf.image.resize(
            tf.image.decode_jpeg(tf.io.read_file(p), channels=3),
            [h, w]), tf.float32))[np.newaxis]
    prep_t = tf.Variable(tf.cast(prep, tf.float32))

    cam    = gradcam_pp(best_model, prep_t, target_layer)
    cam_up = tf.image.resize(cam[..., np.newaxis], [h, w]).numpy()[:, :, 0]
    hmap   = (cm_lib.jet(cam_up)[:, :, :3] * 255).astype(np.uint8)
    overlay = (0.55 * orig + 0.45 * hmap).astype(np.uint8)

    pred_idx  = np.argmax(best_model(prep_t, training=False).numpy())
    pred_name = class_names[pred_idx]
    correct   = sample_labels[idx] == pred_name

    axes[idx, 0].imshow(orig)
    axes[idx, 0].set_title(f'Original: {sample_labels[idx]}', fontsize=8)
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(cam_up, cmap='jet')
    axes[idx, 1].set_title('Grad-CAM++ heatmap', fontsize=8)
    axes[idx, 1].axis('off')

    axes[idx, 2].imshow(overlay)
    axes[idx, 2].set_title(
        f'Pred: {pred_name}', fontsize=8,
        color='green' if correct else 'red')
    axes[idx, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(BASE_RESULT_DIR, 'gradcam_pp.png'), dpi=200)
plt.show()

# Free GPU memory for next cell
del best_model
gc.collect()

In [ ]:
# ========== 16. INFERENCE SPEED BENCHMARK ========== #

import gc

# Explicitly delete any lingering model references from previous cells,
# then clear the Keras session to free all GPU memory
for _var in ['best_model', 'gap_model', 'vis_model']:
    if _var in dir():
        del _var
gc.collect()
tf.keras.backend.clear_session()

best_run   = max(all_runs_results, key=lambda r: r['accuracy'])
best_model = load_model(
    best_run['best_model_path'], custom_objects=CUSTOM_OBJECTS)

SPEED_BATCH = 4  # Reduced to 4 to avoid OOM on large EfficientNetB4+FSDA
_, _, test_ds_speed, _ = create_tf_datasets(
    DATA_DIR, INPUT_SHAPE, SPEED_BATCH, seed=42, use_aug=False)

# Warm-up (1 batch only)
for batch_x, _ in test_ds_speed.take(1):
    _ = best_model(batch_x, training=False)

# Benchmark
n_batches, total_imgs, total_t = 0, 0, 0.0
for batch_x, _ in test_ds_speed:
    t0 = time.perf_counter()
    _  = best_model(batch_x, training=False)
    total_t   += time.perf_counter() - t0
    total_imgs += batch_x.shape[0]
    n_batches  += 1

ms_per_img   = total_t / total_imgs * 1000
imgs_per_sec = total_imgs / total_t

print("=" * 50)
print("INFERENCE SPEED BENCHMARK")
print(f"  Images tested  : {total_imgs}")
print(f"  Total time     : {total_t:.3f} s")
print(f"  ms / image     : {ms_per_img:.2f} ms")
print(f"  Images / second: {imgs_per_sec:.1f}")
print("=" * 50)

In [ ]:
# ========== 17. MODEL SUMMARY ========== #

import gc

gc.collect()
tf.keras.backend.clear_session()

best_run   = max(all_runs_results, key=lambda r: r['accuracy'])
best_model = load_model(
    best_run['best_model_path'], custom_objects=CUSTOM_OBJECTS)

total_params     = best_model.count_params()
trainable_params = sum(tf.size(v).numpy() for v in best_model.trainable_weights)
frozen_params    = total_params - trainable_params

print("=" * 60)
print("MODEL SUMMARY")
print(f"  Architecture : EfficientNetB4 + FSDABlock")
print(f"  Total params : {total_params:,}")
print(f"  Trainable    : {trainable_params:,}")
print(f"  Frozen       : {frozen_params:,}")
print(f"  Input shape  : {INPUT_SHAPE}")
print(f"  FSDA reduction   : {FSDA_REDUCTION}")
print(f"  FSDA spatial_ks  : {FSDA_SPATIAL_KS}")
print("=" * 60)

print("\nFSDA Block layers:")
for layer in best_model.layers:
    if 'fsda' in layer.name:
        print(f"  {layer.name:40s}  params={layer.count_params():,}")

print("\nClassification Head layers:")
for lname in ['gap', 'head_bn', 'head_dense', 'head_dropout', 'predictions']:
    try:
        ly = best_model.get_layer(lname)
        # Keras 3 removed .output_shape on layer objects;
        # retrieve it from the model's output tensors instead.
        try:
            out_shape = ly.output.shape
        except AttributeError:
            out_shape = 'N/A'
        print(f"  {ly.name:40s}  output={out_shape}  params={ly.count_params():,}")
    except ValueError:
        pass

In [ ]:
# ========== 18. EXPORT FINAL REPORT ========== #

report_path = os.path.join(BASE_RESULT_DIR, 'final_report.csv')
with open(report_path, 'w', newline='') as f:
    fields = [
        'strategy_key', 'strategy_label', 'unfreeze_blocks',
        'lr', 'use_aug', 'batch_size',
        'fsda_reduction', 'fsda_spatial_ks',
        'run', 'seed',
        'accuracy', 'precision', 'recall', 'f1_score',
    ]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for r in all_runs_results:
        writer.writerow({
            'strategy_key':    STRATEGY_KEY,
            'strategy_label':  STRATEGY_LABEL,
            'unfreeze_blocks': str(UNFREEZE_BLOCKS),
            'lr':              LR,
            'use_aug':         USE_AUG,
            'batch_size':      BATCH_SIZE,
            'fsda_reduction':  FSDA_REDUCTION,
            'fsda_spatial_ks': FSDA_SPATIAL_KS,
            'run':             r['run'],
            'seed':            r['seed'],
            'accuracy':        f"{r['accuracy']:.6f}",
            'precision':       f"{r['precision']:.6f}",
            'recall':          f"{r['recall']:.6f}",
            'f1_score':        f"{r['f1_score']:.6f}",
        })

acc_mean = np.mean([r['accuracy'] for r in all_runs_results])
acc_std  = np.std( [r['accuracy'] for r in all_runs_results])
f1_mean  = np.mean([r['f1_score'] for r in all_runs_results])
f1_std   = np.std( [r['f1_score'] for r in all_runs_results])

print("=" * 60)
print("FINAL REPORT")
print(f"  {STRATEGY_LABEL}")
print(f"  Accuracy : {acc_mean:.4f} +/- {acc_std:.4f}")
print(f"  F1-Score : {f1_mean:.4f} +/- {f1_std:.4f}")
print(f"  Saved    : {report_path}")
print("=" * 60)

print("\nAll outputs saved to:", BASE_RESULT_DIR)
for fname in sorted(os.listdir(BASE_RESULT_DIR)):
    print(f"  {fname}")

In [ ]:
!zip -r /kaggle/working/finetune_top5_fsda_ldam.zip /kaggle/working/report_EfficientNetB4/finetune_top5_fsda_ldam
